In [0]:
# ============================================================
# NOTEBOOK 6: DASHBOARD PREP
# Team AryaBit | HackBricks 2026
# Purpose: Create Gold aggregation tables for Dashboard
# ============================================================

import logging
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.Dashboard")

CATALOG = "workspace"
SCHEMA  = "default"

def run_dashboard_prep():
    logger.info("Building Dashboard aggregation tables...")

    # ── VIEW 1: RISK TIER SUMMARY ─────────────────────────────
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_dashboard_risk_summary AS
        SELECT
            intervention_tier,
            COUNT(*)                                         AS student_count,
            ROUND(AVG(risk_score), 4)                       AS avg_risk_score,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
        FROM {CATALOG}.{SCHEMA}.gold_at_risk_students
        GROUP BY intervention_tier
        ORDER BY avg_risk_score DESC
    """)
    logger.info("✅ gold_dashboard_risk_summary created")

    # ── VIEW 2: FAIRNESS SUMMARY ──────────────────────────────
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_dashboard_fairness AS
        SELECT
            group_type,
            group_label,
            group_size,
            actual_dropout_rate,
            predicted_dropout_rate,
            demographic_parity,
            equal_opportunity,
            false_positive_rate,
            CASE
                WHEN ABS(demographic_parity - 
                    AVG(demographic_parity) OVER(PARTITION BY group_type)) > 0.05
                THEN 'DISPARITY DETECTED'
                ELSE 'FAIR'
            END AS fairness_verdict
        FROM {CATALOG}.{SCHEMA}.gold_fairness_audit
    """)
    logger.info("✅ gold_dashboard_fairness created")

    # ── VIEW 3: INTERVENTION PIPELINE ────────────────────────
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_dashboard_interventions AS
        SELECT
            intervention_tier,
            bias_flag,
            COUNT(*)                          AS student_count,
            ROUND(AVG(risk_score), 4)         AS avg_risk_score,
            ROUND(MIN(risk_score), 4)         AS min_risk_score,
            ROUND(MAX(risk_score), 4)         AS max_risk_score,
            SUM(CASE WHEN gender = 0 THEN 1 ELSE 0 END) AS female_count,
            SUM(CASE WHEN gender = 1 THEN 1 ELSE 0 END) AS male_count,
            SUM(CASE WHEN debtor = 1 THEN 1 ELSE 0 END) AS debtor_count,
            SUM(CASE WHEN scholarship_holder = 1 
                THEN 1 ELSE 0 END)            AS scholarship_count
        FROM {CATALOG}.{SCHEMA}.gold_at_risk_students
        GROUP BY intervention_tier, bias_flag
        ORDER BY avg_risk_score DESC
    """)
    logger.info("✅ gold_dashboard_interventions created")

    # ── VIEW 4: RISK SCORE DISTRIBUTION ──────────────────────
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_dashboard_score_dist AS
        SELECT
            ROUND(risk_score, 1)              AS risk_bucket,
            COUNT(*)                          AS student_count,
            intervention_tier
        FROM {CATALOG}.{SCHEMA}.gold_at_risk_students
        GROUP BY ROUND(risk_score, 1), intervention_tier
        ORDER BY risk_bucket DESC
    """)
    logger.info("✅ gold_dashboard_score_dist created")

    # ── VIEW 5: FINANCIAL STRESS IMPACT ──────────────────────
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_dashboard_financial AS
        SELECT
            CASE debtor 
                WHEN 1 THEN 'Has Debt'
                ELSE 'No Debt'
            END                               AS debt_status,
            CASE scholarship_holder
                WHEN 1 THEN 'Has Scholarship'
                ELSE 'No Scholarship'
            END                               AS scholarship_status,
            COUNT(*)                          AS student_count,
            ROUND(AVG(risk_score), 4)         AS avg_risk_score,
            SUM(CASE WHEN intervention_tier = 'HIGH' 
                THEN 1 ELSE 0 END)            AS high_risk_count
        FROM {CATALOG}.{SCHEMA}.gold_at_risk_students
        GROUP BY debtor, scholarship_holder
        ORDER BY avg_risk_score DESC
    """)
    logger.info("✅ gold_dashboard_financial created")

    # ── FINAL SUMMARY ─────────────────────────────────────────
    print("\n" + "="*60)
    print("DASHBOARD TABLES READY")
    print("="*60)
    
    tables = [
        "gold_dashboard_risk_summary",
        "gold_dashboard_fairness",
        "gold_dashboard_interventions",
        "gold_dashboard_score_dist",
        "gold_dashboard_financial"
    ]
    
    for t in tables:
        count = spark.sql(
            f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.{t}"
        ).collect()[0][0]
        print(f"✅ {t}: {count} rows")

    logger.info("Dashboard Prep COMPLETE ✅")

run_dashboard_prep()